# GPA778 - Laboratory 2
# Part C - Perception III: ROS2 Actions and Point Cloud Filtering

In the previous lab, you implemented a 3D scanning system using a 2D LiDAR mounted on a servo and controlled by **ROS2 services**. Today, you will discover **ROS2 actions**, a mechanism better suited to long-running operations, and learn how to filter point clouds.

**Lab objectives:**
- Understand the difference between ROS2 services and actions
- Understand the role of ROS2 executors and callback groups
- Integrate a state machine into an action's execution callback
- Integrate a processing state into the state machine
- Implement point cloud filters with Open3D
- Configure filtering parameters through a ROS2 action

**Estimated duration:** 2 h 30 min


---
## Section 1: Review of ROS2 Services

### What is a ROS2 service?

A ROS2 service is a **synchronous**, **request-response** communication mechanism.

**Characteristics:**
- The client sends a request and **waits** for the response
- The client is blocked for the entire execution time
- Suitable for **short** operations (<1 second)

**Example from Lab 2b: SetServoPos**

You used the `SetServoPos` service to control the servo position:

```python
# Create the client
self.set_servo_pos_srv = self.create_client(SetServoPos, 'set_servo_pos')

# Send a request
req = SetServoPos.Request()
req.position = 2000
future = self.set_servo_pos_srv.call_async(req)
future.add_done_callback(self._on_servo_response)
```

### Limitations of services

**Think about it:**
> "What happens if a service takes 10 seconds to execute? Can the client tell how far execution has progressed?"

**Answer:** No! The client waits silently until it receives the final response. It gets **no feedback** during execution.


---
## Section 2: Introduction to ROS2 Actions

### The problem with services for long-running operations

Imagine a 3D scan that takes **8 seconds** to execute:
- The client is **blocked** for 8 seconds
- There is no way to know whether the scan is at 10%, 50%, or 90%
- It is impossible to **cancel** the scan while it is running
- It is impossible to know whether the server has crashed or is simply slow

**Solution: ROS2 Actions**

### What is a ROS2 action?

An action is an **asynchronous** communication mechanism for **long-running** operations, with:
- **Periodic feedback** during execution
- The ability to **cancel** the operation
- A final result when it finishes

### Service vs. Action comparison

| Aspect | Service | Action |
|--------|---------|--------|
| Duration | Short (<1s) | Long (>1s) |
| Feedback | ❌ No | ✅ Yes (periodic) |
| Cancellation | ❌ No | ✅ Yes |
| Result | Immediate | Deferred |
| Client blocking | ✅ Blocking | ❌ Asynchronous |
| Example | `SetServoPos` | `StartScan` |

### Structure of an action

A ROS2 action contains **three sections**:

#### 1. **Goal**
The parameters of the operation to perform (equivalent to a service request)

#### 2. **Result**
The final result returned when the operation finishes (equivalent to a service response)

#### 3. **Feedback**
Information sent **periodically** during execution (e.g., percentage completed)

### Example: StartScan.action file

**Reading exercise:**
> "Open the file `/home/ubuntu/ros2_ws/src/jerro/jerro_msgs/action/StartScan.action` and identify its three sections."


In [ ]:
# Let's view the structure of StartScan.action
!cat /home/ubuntu/ros2_ws/src/jerro/jerro_msgs/action/StartScan.action


### Analysis of StartScan.action

**Goal:**
- `start`: Boolean to start the scan
- **New filtering parameters**: `enable_filtering`, `enable_statistical_filter`, etc.

**Result:**
- `point_cloud`: The resulting 3D point cloud
- `success`: Boolean indicating whether the scan succeeded
- `message`: Descriptive message

**Feedback (periodic):**
- `progress`: Float between 0.0 and 1.0 indicating scan progress

### Sequence diagram

```
Client                    Action Server
  |                             |
  |──────(Goal)───────────────> |
  |                             | goal_callback()
  | <───(Accept/Reject)────────|
  |                             | execute_callback()
  | <───(Feedback 10%)─────────|
  |                             |
  | <───(Feedback 50%)─────────|
  |                             |
  | <───(Feedback 90%)─────────|
  |                             |
  | <───(Result + Success)─────|
  |                             |
```


---
## Section 3: The StartScan Action in lidar_traitement.py
### Creating an Action Server

In the `lidar_traitement.py` file, an Action Server is created as follows:

```python
from rclpy.action import ActionServer, GoalResponse, CancelResponse
from jerro_msgs.action import StartScan

self._action_server = ActionServer(
    self,                                    # The ROS2 node
    StartScan,                               # Action type
    '/perception/start_scan',                # Topic name
    execute_callback=self.execute_scan_callback,
    goal_callback=self.goal_callback,
    cancel_callback=self.cancel_callback,
    callback_group=self._group,              # See Section 4
)
```

### The three callbacks

#### 1. **goal_callback**: Accept or reject the goal

```python
def goal_callback(self, goal_request):
    if not self.set_servo_pos_srv.service_is_ready():
        return GoalResponse.REJECT  # Servo is not ready
    if self.actual_scan is None:
        return GoalResponse.REJECT  # No scan received
    return GoalResponse.ACCEPT      # OK, accept it
```

#### 2. **cancel_callback**: Handle cancellation requests

```python
def cancel_callback(self, goal_handle):
    self.get_logger().info("Cancellation requested")
    return CancelResponse.ACCEPT
```

#### 3. **execute_callback**: Execute the action using a loop

This is the central callback. It **starts when the goal is accepted** and **ends when it returns a result**:

```python
def execute_scan_callback(self, goal_handle):
    # Initialization...
    self._scan_state.set_next_state(ScanState.MOVE_TO_START)

    while True:                                    # ← main loop
        match self._scan_state.get_actual_state():
            case ScanState.FINISHING:
                self._finalize_scan()
                result = StartScan.Result()
                result.success = True
                goal_handle.succeed()
                return result                      # ← callback ends (success)

            case ScanState.WAITING_SERVO:
                if prev == ScanState.CANCELLING:
                    goal_handle.canceled()
                    return result                  # ← callback ends (cancellation)
            # ...
```

**Key points:**
- The callback **loops** until the scan is complete
- It ends with `return result` (success or cancellation)
- `goal_handle` is available locally (there is no need to store it as a class attribute)
- Feedback is published directly from the loop: `goal_handle.publish_feedback(feedback)`

### Comprehension questions

1. **Which line ends the callback on success?**
   - Answer: `return result` in the `FINISHING` case, after `goal_handle.succeed()`

2. **What happens if `goal_callback` returns `GoalResponse.REJECT`?**
   - Answer: `execute_callback` is never called

3. **Compare with v1: in `lidar_traitement_v1.py`, what controlled the state machine?**
   - Answer: A timer called `state_machine_tick()` every 50 ms. Here, it is the `while True` loop.


---
## Section 4: Executors and Callback Groups

### What is an executor?

An **executor** is the ROS2 component responsible for **executing callbacks**: timers, subscribers, services, actions, and so on.

When you write `rclpy.spin(node)`, you implicitly use a **single-threaded** executor that processes all callbacks **one after another**.

### The problem: execute_callback blocks

`execute_scan_callback` contains a `while True` loop that **does not end until the scan is complete** (several seconds). During this time, the thread is busy.

While inside this `while True`, the loop waits for `self._servo_done` to become `True`:

```python
case ScanState.WAITING_SERVO:
    if not self._servo_done:   # ← wait here (busy-wait)
        continue
```

What sets `_servo_done = True`? `_on_servo_response`, a **callback** called when the servo confirms its position:

```python
def _on_servo_response(self, _future):
    self._servo_done = True     # ← must run during the while True loop!
```

### Deadlock with a single thread

```
Single thread:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  1. execute_scan_callback() starts and enters while True     │
│     → sends servo command (_send_servo_pos)                 │
│     → WAITING_SERVO: loops while waiting for _servo_done... │
│         while not self._servo_done: continue  ←── BLOCKED!  │
│                                                             │
│  2. _on_servo_response() wants to run...                    │
│     → IMPOSSIBLE! The thread is occupied by while True     │
│                                                             │
│  3. _servo_done will NEVER be set to True                   │
│     → DEADLOCK                                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### The solution: MultiThreadedExecutor

With multiple threads, callbacks can run in parallel:

```
Thread 1:  execute_scan_callback()  ──[while True... WAITING_SERVO...]──
                                                                         ↑ unblocked!
Thread 2:                    _on_servo_response()  →  _servo_done = True
Thread 3:         scan_callback()  (continuously receives LiDAR data)
```

**Implementation:**

```python
from rclpy.executors import MultiThreadedExecutor

def main(args=None):
    rclpy.init(args=args)
    node = Lidar3D()
    executor = MultiThreadedExecutor()   # ← multiple threads
    executor.add_node(node)
    executor.spin()                      # ← replaces rclpy.spin()
```

### Callback Groups

However, `MultiThreadedExecutor` alone is not enough! ROS2 uses **Callback Groups** to decide which callbacks can run in parallel.

#### MutuallyExclusiveCallbackGroup (default behavior)

Callbacks in the **same group** **never run in parallel**—only one at a time:

```
Group A (MutuallyExclusive):
  callback_1 ──────────────────┐
  callback_2                   └──────────────── (must wait)
```

If `execute_scan_callback` and `cancel_callback` belong to the **same** `MutuallyExclusiveCallbackGroup`, cancellation can never be handled while the scan is running.

#### ReentrantCallbackGroup

Callbacks in the **same group** **can** run in parallel:

```
Group B (Reentrant):
  execute_callback ──────────────────────────────────
  cancel_callback       ──────                       (can run at the same time!)
```

**In practice:**

Each callback created separately (using `create_subscription`, `create_timer`, etc.) receives **its own** `MutuallyExclusiveCallbackGroup` by default. These **different** groups can therefore still run in parallel with `MultiThreadedExecutor`.

The specific problem in our case is that **multiple action callbacks** (execute + cancel) would belong to the same default group. Putting them in a `ReentrantCallbackGroup` ensures that cancellation can interrupt execution:

```python
from rclpy.callback_groups import ReentrantCallbackGroup

self._group = ReentrantCallbackGroup()

self._action_server = ActionServer(
    self,
    StartScan,
    '/perception/start_scan',
    execute_callback=self.execute_scan_callback,
    goal_callback=self.goal_callback,
    cancel_callback=self.cancel_callback,
    callback_group=self._group,          # ← Reentrant: parallel execution is possible
)
```

### Summary table

| | `rclpy.spin()` | `MultiThreadedExecutor` alone | `MultiThreadedExecutor` + `ReentrantCallbackGroup` |
|-|----------------|------------------------------|------------------------------------------------------|
| **Threads** | 1 | N | N |
| **Blocking while True** | ❌ Deadlock | ✅ OK | ✅ OK |
| **Cancellation during scan** | ❌ Impossible | ⚠️ Partial | ✅ Guaranteed |
| **scan_callback in parallel** | ❌ | ✅ | ✅ |

### Comprehension questions

1. **If `rclpy.spin()` is used with the while True loop, what happens in WAITING_SERVO?**
   - Answer: Deadlock—the while True loop monopolizes the thread, `_on_servo_response` can never run, and `_servo_done` remains False forever.

2. **Without `ReentrantCallbackGroup`, what happens if cancellation is requested during the scan?**
   - Answer: `cancel_callback` cannot run while `execute_scan_callback` is running (same `MutuallyExclusive` group).

3. **`lidar_traitement.py` used `rclpy.spin()`. Why did it not deadlock?**
   - Answer: The state machine was inside a timer—each call to `state_machine_tick()` finished in a few milliseconds. There was no blocking loop, so the thread was released quickly.


---
## Section 5: State Machine Inside the while True Loop

### Architecture: timer vs. while True

In `lidar_traitement_v1.py`, the state machine was driven by a **timer**:

```
Timer (every 50 ms) ──> state_machine_tick()  ──> advances one state
                  ↑___________________________|  (short callback)
```

In `lidar_traitement.py`, it is integrated into the **action's while loop**:

```
execute_scan_callback() ─────────────────────────────────────────────>
   while True:                                                         |
     MOVE_TO_START → WAITING_SERVO → SCANNING → ... → FINISHING        |
                                                         return result |
```

**Advantage:** All scan logic is in one place. It is more readable and direct.

### Current ScanState states

```python
class ScanState:
    IDLE = 0          # Not used in the loop (initial state)
    MOVE_TO_START = 1 # Move servo to the starting position
    WAITING_SERVO = 2 # Wait for servo response (busy-wait)
    SCANNING = 3      # Scan and publish feedback
    RESETTING = 4     # Return servo to neutral position
    FINISHING = 5     # Finalize, publish cloud, return result
    CANCELLING = 6    # Cancellation in progress
```

### Transition diagram

```
(goal received) → MOVE_TO_START
                       ↓ (_send_servo_pos)
                  WAITING_SERVO ←────────────────────────────┐
                       ↓ (_servo_done = True)                │
                   SCANNING ──(cancel)──> CANCELLING         │
                       │                      │ (_send_servo_pos(1500))
                       │ (end of travel)       └──────────────┘
                       ↓
                   RESETTING
                       ↓ (_send_servo_pos(1500))
                  WAITING_SERVO
                       ↓
                   FINISHING
                       ↓
                 return result
```

### Adding the PROCESSING state

To add filtering, a `PROCESSING` state must be inserted between `WAITING_SERVO` (after RESETTING) and `FINISHING`:

```
... RESETTING
       ↓
  WAITING_SERVO
       ↓ (if enable_filtering)
  PROCESSING          ↓ (otherwise)
       └────────────> FINISHING
                          ↓
                    return result
```

**Question:** How can the state machine transition conditionally to PROCESSING or directly to FINISHING?

**Hint:** Use `goal_handle.request.enable_filtering`

**Note:** In the `while True` version, `goal_handle` is available as a **local variable** in the callback (not as a class attribute). You can use it directly in the loop.


---
## Section 6: Open3D Filters — Theory

### Introduction to filtering

#### Why filter point clouds?

Raw point clouds from a LiDAR often contain:
- **Noise:** Incorrect points caused by sensor limitations
- **Outliers:** Anomalous points far from their neighbors
- **Excessive density:** Too many points for real-time processing
- **Points outside the area of interest:** Points beyond the robot's immediate surroundings

#### Sources of noise in a LiDAR scan

- **Multiple reflections:** The laser bounces more than once
- **Reflective surfaces:** Mirrors and windows
- **Dust and particles:** In the air
- **Measurement errors:** Sensor inaccuracy
- **Mechanical vibrations:** From the servo or robot

### Filter 1: Crop (Bounding Box)

#### Principle

Crop the cloud to keep only points **inside a defined volume** (bounding box).

Useful for:
- Limiting the area of interest (e.g., within 5 m of the robot)
- Removing the floor or ceiling
- Reducing the amount of data to process

#### Parameters

- `x_min`, `x_max`: X limits (front/back of the robot)
- `y_min`, `y_max`: Y limits (left/right)
- `z_min`, `z_max`: Z limits (bottom/top)

#### Visualization

```
         z ↑
           │    crop_z_max
           │   ┌───────────┐
           │   │           │
           │   │   Kept    │
           │   │   area    │
           │   │           │
           │   └───────────┘
           │    crop_z_min
           └──────────────────> x
```

#### Open3D method

```python
bbox = o3d.geometry.AxisAlignedBoundingBox(
    min_bound=[x_min, y_min, z_min],
    max_bound=[x_max, y_max, z_max]
)
cropped_cloud = cloud.crop(bbox)
```

#### Reference

**Open3D documentation:**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

#### Questions

1. **If the robot is 0.3 m tall, what value of `z_min` can remove the floor?**
   - Answer: Approximately `z_min = 0.05` to `0.1` to avoid cropping the robot itself

2. **Why apply cropping first?**
   - Answer: To reduce the amount of data **before** the other computationally expensive filters


### Filter 2: Statistical Outlier Removal

#### Principle

Removes **statistical outliers** based on their distance from neighboring points.

**Algorithm:**
1. For each point, calculate the mean distance to its **K nearest neighbors**
2. Calculate the **mean** and **standard deviation** of all these distances
3. Remove points whose mean distance > `mean + std_ratio × standard deviation`

#### Visualization

```
    ●                     ●  ← Isolated outlier (will be removed)

         ● ● ●
        ● ● ● ●           ← Dense cluster (will be kept)
         ● ● ●

                 ●        ← Outlier (will be removed)
```

#### Parameters

- **`nb_neighbors`**: Number of neighbors to consider (e.g., 20)
  - Higher → more robust but slower
  - Lower → faster but less reliable

- **`std_ratio`**: Threshold as a multiple of the standard deviation (e.g., 2.0)
  - Higher → **more permissive** (keeps more points)
  - Lower → **stricter** (removes more points)

#### Open3D method

```python
filtered_cloud, indices = cloud.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=2.0
)
# filtered_cloud: filtered cloud
# indices: list of indices of the retained points
```

#### Reference

**Open3D documentation:**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal

#### Questions

1. **Does a higher `std_ratio` make the filter stricter or more permissive?**
   - Answer: **More permissive** (keeps more points because the threshold is higher)

2. **Why might a `nb_neighbors` value that is too small remove valid points?**
   - Answer: With few neighbors, the mean distance is statistically less reliable

3. **When should you use this filter?**
   - Answer: To remove **noise** and anomalous **isolated points**


### Filter 3: Voxel Downsampling

#### Principle

Reduces point cloud **density** using a 3D grid of **voxels** (cubes).

**Algorithm:**
1. Divide space into a 3D grid of fixed-size voxels
2. For each voxel containing points:
   - Calculate the **centroid** (mean) of the points in the voxel
   - Replace all points with this single centroid

**Result:** At most one point per voxel

#### 2D visualization

```
Before:                  After:
┌────┬────┬────┐        ┌────┬────┬────┐
│ ●● │ ●  │    │        │ ●  │ ●  │    │
│ ●  │    │  ● │   →    │    │    │  ● │
├────┼────┼────┤        ├────┼────┼────┤
│ ●●●│ ●● │    │        │ ●  │ ●  │    │
│ ●  │ ●  │    │        │    │    │    │
└────┴────┴────┘        └────┴────┴────┘

8 points → 4 points
```

#### Parameters

- **`voxel_size`**: Voxel size in meters (e.g., 0.02 = 2 cm)
  - Larger → **fewer points** (greater reduction)
  - Smaller → **more points** (less reduction)

#### Open3D method

```python
downsampled_cloud = cloud.voxel_down_sample(voxel_size=0.02)
```

#### Reference

**Open3D documentation:**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling

#### Questions

1. **Will a `voxel_size` of 0.05 m instead of 0.02 m produce more or fewer points?**
   - Answer: **Fewer points** (larger voxels = fewer voxels = fewer points)

2. **What is the advantage of voxel downsampling over simple random sampling?**
   - Answer: It preserves a **uniform distribution** in space, avoiding empty regions

3. **When should you use this filter?**
   - Answer: To reduce **density** and speed up subsequent processing


### Order of filter application

Filter order is **important** for efficiency and result quality.

#### Recommended order

1. **Crop** first
   - Why? Reduces the data volume **before** expensive filters
   - Example: 100,000 points → 50,000 points

2. **Statistical Outlier Removal** second
   - Why? Removes noise from data that has already been reduced
   - Example: 50,000 points → 45,000 points

3. **Voxel Downsampling** last
   - Why? Optimizes final density after cleaning
   - Example: 45,000 points → 10,000 points

#### Filtering pipeline

```
Raw cloud (100k points, noisy, large area)
    ↓
[CROP] → Restricts to the area of interest
    ↓
Cropped cloud (50k points, noisy, reduced area)
    ↓
[STATISTICAL] → Removes outliers
    ↓
Cleaned cloud (45k points, clean, reduced area)
    ↓
[VOXEL] → Reduces density
    ↓
Final cloud (10k points, clean, optimized)
```

#### Summary table

| Filter | Purpose | Reduces point count? | Computational cost |
|--------|---------|----------------------|--------------------|
| Crop | Restricts the area | Yes (a lot) | Low |
| Statistical | Removes noise | Yes (moderately) | High |
| Voxel | Reduces density | Yes (a lot) | Moderate |


---
## Section 7: Practical Exercises

### Environment setup

Before starting, update your workspace with the new action definition:


In [ ]:
%%bash
cd ~/ros2_ws/src/jerro
git pull
cd ~/ros2_ws
colcon build --packages-select jerro_msgs
source install/setup.bash

### Exercise 1: Understanding the new action definition

**Objective:** Analyze the modified StartScan.action definition

**Instructions:**
1. Open the `StartScan.action` file
2. Identify all new parameters added to the Goal
3. For each parameter, explain its purpose using the Open3D documentation

**Questions:**
- Why are some parameters `bool` and others `float32`?
- How many parameters were added in total?
- What is the suggested default value for `voxel_size`?


### Exercise 2: Replacing the timer with while True (30 min)

**Objective:** Modify `execute_scan_callback` so that it directly contains the state machine

**File to modify:** `lidar_traitement.py`

**Context:** The current version of the file still has the timer commented out. Your task is to rewrite `execute_scan_callback` using a `while True` loop.

**TODO 1: Configure ReentrantCallbackGroup**

```python
# TODO: Import ReentrantCallbackGroup
from rclpy.callback_groups import ___

# TODO: Create the group in __init__
self._group = ___

# TODO: Assign it to the action server
self._action_server = ActionServer(
    ...
    callback_group=___,
)
```

**TODO 2: Rewrite execute_scan_callback**

```python
def execute_scan_callback(self, goal_handle):  # ← no longer async
    # TODO: Initialize the scan (servo_pos, global_scan_3d, servo_done, initial state)

    while True:
        match self._scan_state.get_actual_state():

            case ScanState.MOVE_TO_START:
                # TODO: Send the servo to the starting position

            case ScanState.WAITING_SERVO:
                # TODO: Busy-wait if _servo_done is False
                # TODO: Based on the previous state, transition to SCANNING, FINISHING, or handle cancellation

            case ScanState.SCANNING:
                # TODO: Check whether cancellation was requested (goal_handle.is_cancel_requested)
                # TODO: Collect the scan and publish feedback
                # TODO: Decrement the servo position and send it if necessary

            case ScanState.RESETTING:
                # TODO: Send the servo to the neutral position (1500)

            case ScanState.CANCELLING:
                # TODO: Send the servo to the neutral position

            case ScanState.FINISHING:
                # TODO: Finalize the scan, create the result, call goal_handle.succeed()
                # TODO: return result  ← ends the callback
```

**TODO 3: Use MultiThreadedExecutor in main()**

```python
def main(args=None):
    rclpy.init(args=args)
    node = Lidar3D()
    # TODO: Create a MultiThreadedExecutor, add the node, and call spin()
```

**Hints:**
- `goal_handle` is available as a local variable in `execute_scan_callback`
- For busy-waiting: `if not self._servo_done: continue`
- The callback ends with `return result` (no Future or `set_result`)
- Remember to reset `_servo_done = False` before sending a servo command


### Exercise 3: Adding the PROCESSING state

**Objective:** Integrate a new state into the state machine

**File to modify:** `/home/ubuntu/ros2_ws/src/jerro_perception/jerro_perception/lidar_traitement.py`

**TODO 1: Modify the ScanState class**

```python
class ScanState:
    IDLE = 0
    MOVE_TO_START = 1
    WAITING_SERVO = 2
    SCANNING = 3
    RESETTING = 4
    # TODO: Add the PROCESSING state = ___
    FINISHING = ___  # TODO: Adjust the value
    CANCELLING = ___  # TODO: Adjust the value
```

**TODO 2: Add the PROCESSING case to state_machine_tick()**

```python
case ScanState.PROCESSING:
    # TODO: Call the self._process_point_cloud() method
    # TODO: Transition to the FINISHING state
    pass
```

**TODO 3: Modify the WAITING_SERVO case for the conditional transition**

```python
case ScanState.WAITING_SERVO:
    if not self._servo_done:
        return
    prev = self._scan_state.get_previous_state()
    if prev == ScanState.RESETTING:
        # TODO: Check whether filtering is enabled in the goal
        # TODO: If enabled, transition to PROCESSING; otherwise, to FINISHING
        pass
```

**Hint:** Use `self._goal_handle.request.enable_filtering`


### Exercise 4: Implementing the filters

**Objective:** Create the three filtering methods

**File to modify:** `lidar_traitement.py`

#### Filter 1: _apply_crop

```python
def _apply_crop(self, cloud, x_range, y_range, z_range):
    """
    TODO: Crop the cloud to a bounding box

    Args:
        cloud: Open3D point cloud
        x_range: [min, max] along x
        y_range: [min, max] along y
        z_range: [min, max] along z

    Returns:
        Cropped cloud

    Reference: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

    Hints:
    - Create an AxisAlignedBoundingBox with min_bound and max_bound
    - Use the cloud's .crop() method
    - Log the number of points before and after
    """
    # TODO: Your code here
    pass
```

#### Filter 2: _apply_statistical_filter

```python
def _apply_statistical_filter(self, cloud, nb_neighbors, std_ratio):
    """
    TODO: Statistical filter to remove outliers

    Args:
        cloud: Open3D point cloud
        nb_neighbors: Number of neighbors to consider
        std_ratio: Standard deviation ratio

    Returns:
        Filtered cloud

    Reference: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal

    Hints:
    - Use the .remove_statistical_outlier() method
    - This method returns TWO values: (filtered_cloud, indices)
    - Log the number of points before and after
    """
    # TODO: Your code here
    pass
```

#### Filter 3: _apply_voxel_downsampling

```python
def _apply_voxel_downsampling(self, cloud, voxel_size):
    """
    TODO: Downsample the cloud using a voxel grid

    Args:
        cloud: Open3D point cloud
        voxel_size: Voxel size in meters

    Returns:
        Downsampled cloud

    Reference: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling

    Hints:
    - Use the .voxel_down_sample() method
    - Log the number of points before and after
    """
    # TODO: Your code here
    pass
```


### Exercise 5: Integration into _process_point_cloud()

**Objective:** Create the method that orchestrates all filters

```python
def _process_point_cloud(self):
    """
    TODO: Apply the filters configured in the goal to the global cloud

    Application order:
    1. Crop (reduces the data volume)
    2. Statistical outlier removal (removes noise)
    3. Voxel downsampling (optimizes final density)

    Each filter should only be applied if its activation Boolean is True
    """
    if self._goal_handle is None:
        return

    goal = self._goal_handle.request
    self.get_logger().info("Starting point cloud processing...")

    filtered_cloud = self.global_scan_3d

    # TODO 1: Apply cropping if goal.enable_crop is True
    # Hints:
    #   - Create the ranges: x_range = [goal.crop_x_min, goal.crop_x_max]
    #   - Call self._apply_crop(filtered_cloud, x_range, y_range, z_range)
    #   - Assign the result to filtered_cloud

    # TODO 2: Apply the statistical filter if goal.enable_statistical_filter is True
    # Hints:
    #   - Get the parameters: goal.statistical_nb_neighbors, goal.statistical_std_ratio
    #   - Call self._apply_statistical_filter(...)

    # TODO 3: Apply voxel downsampling if goal.enable_voxel_downsampling is True
    # Hints:
    #   - Get the parameter: goal.voxel_size
    #   - Call self._apply_voxel_downsampling(...)

    # Replace the global cloud with the filtered cloud
    self.global_scan_3d = filtered_cloud
    self.get_logger().info("Point cloud processing complete")
```


### Exercise 6: Testing and tuning

**Objective:** Test your implementation and find the optimal parameters

#### Step 1: Build


In [ ]:
%%bash
cd ~/ros2_ws
colcon build --packages-select jerro_perception
source install/setup.bash

#### Step 2: Launch the node

```bash
ros2 run jerro_perception lidar_traitement
```

#### Step 3: Send a goal with filtering enabled

**Terminal 2:**
```bash
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan \
  "{
    start: true,
    enable_filtering: true,
    enable_crop: true,
    crop_x_min: -2.5, crop_x_max: 2.5,
    crop_y_min: -2.5, crop_y_max: 2.5,
    crop_z_min: -0.5, crop_z_max: 1.0,
    enable_statistical_filter: true,
    statistical_nb_neighbors: 20,
    statistical_std_ratio: 2.0,
    enable_voxel_downsampling: true,
    voxel_size: 0.02
  }" \
  --feedback
```

#### Step 4: Visualization in RViz

**Terminal 3:**
```bash
rviz2
```

- Add a `PointCloud2` display
- Topic: `/perception/lidar_pc2`
- Fixed Frame: `base_link`

#### Step 5: Experimentation

Test different configurations:

1. **No filtering:** `enable_filtering: false`
2. **Crop only:** Only enable_crop
3. **Statistical only:** Only enable_statistical_filter
4. **All filters:** Enable all filters
5. **Aggressive parameters:** `voxel_size: 0.05`, `std_ratio: 1.0`

**Analysis questions:**
- How many points are there before and after filtering?
- Which filter reduces the number of points the most?
- Which parameters provide the best quality/density trade-off for your environment?


---
## Section 8: Laboratory 2 Review Questions

Answer the following questions to check your understanding:

### Question 1: Actions vs. Services

**Why use an action rather than a service for StartScan?**


---

### Question 2: Executor and Callback Groups

**Explain why `lidar_traitement.py` needs both `MultiThreadedExecutor` AND `ReentrantCallbackGroup`.**

Your answer must mention:
- What would happen without `MultiThreadedExecutor` (deadlock in WAITING_SERVO)
- What would happen without `ReentrantCallbackGroup` (cancellation impossible)
- The role of each

---

### Question 3: Parameter Configuration

**What is the advantage of having filtering parameters in the action Goal rather than as [ROS2 parameters](https://docs.ros.org/en/rolling/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Parameters/Understanding-ROS2-Parameters.html) of the node?**

---

### Question 4

In the file `/home/ubuntu/ros2_ws/src/jerro/jerro_description/urdf/jerro.urdf`, the axes of the `base_link_to_servomotor` joint are x=0, y=-1, and z=0. Using the URDF file and the [LiDAR datasheet](https://download-en.slamtec.com/api/download/rplidar-s2m1-Rx-datasheet/2.3?lang=en), explain these values.

```xml
 <joint name="base_link_to_servomotor" type="revolute">
    <parent link="base_link" />
    <child link="servomotor" />
    <origin xyz="0.088 0 0.054" rpy="0 0 0" />
    <axis xyz="0 -1 0"/>
    <limit effort="1000.0" lower="-1" upper="1" velocity="0.5"/>  
    </joint>
```



---

### Question 5: Filter Order

**Explain why cropping is applied first and voxel downsampling last.**



---

### Question 6: Statistical Parameters

**Why might a `nb_neighbors` value that is too small in the statistical filter remove valid points?**


---

### Question 7

**Implement the three filters as presented. Include images showing the differences between the filters.**

---

### Question 8

**To use the camera with the Raspberry Pi, changes had to be made to some files. Using the [v4l2_camera package README](https://docs.ros.org/en/rolling/p/v4l2_camera/), explain what changes were made and why.**

---

### Question 9

**By examining the messages on the `image_raw` and `image_raw/compressed` topics, explain the difference between these two message types and the implications of using each.**

---

### Question 10

**To use the camera, a configuration file was placed in `/home/ubuntu/.ros/camera_info/ost.yaml`. Explain the different parameters in this file and its purpose. Also, what would happen if the file were missing?**


---
## Demonstration to the Lab Instructor

At the end of this lab, you must demonstrate the following:

### Validation checklist

- [ ] **Code builds without errors**
- [ ] **PROCESSING state added** to the state machine
- [ ] **Three filters implemented** (crop, statistical, voxel)
- [ ] **Conditional transition** works (filtering enabled/disabled)
- [ ] **Logs display** the number of points before/after each filter
- [ ] **RViz visualization** shows the difference before/after filtering
- [ ] **Parameters tuned** for your environment
- [ ] **Review questions** answered

### Bonus (optional)

- [ ] **Radius outlier removal** implemented and tested
- [ ] **Quantitative comparison** of different configurations (results table)
- [ ] **Performance analysis** (execution time of each filter)


---
## Useful Resources

### Open3D documentation

- **Main page:** https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html
- **Statistical Outlier Removal:** https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal
- **Radius Outlier Removal:** https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#radius-outlier-removal
- **Downsampling:** https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling
- **Bounding Volume Cropping:** https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

### ROS2 documentation

- **Actions:** https://docs.ros.org/en/rolling/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Actions/Understanding-ROS2-Actions.html
- **Action Servers (Python):** https://docs.ros.org/en/rolling/Tutorials/Intermediate/Writing-an-Action-Server-Client/Py.html
- **Executors and Callback Groups:** https://docs.ros.org/en/rolling/Concepts/Intermediate/About-Executors.html

### Reference code

- `lidar_traitement.py`: Version with Action Server, `while True`, `MultiThreadedExecutor`, `ReentrantCallbackGroup`
- `lidar_traitement_v1.py`: Previous version using services and a timer (for comparison)
